In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
import torch
import math

In [4]:
dataset = load_dataset(
    "text",
    data_files = "data/train.txt",
    split = "train"
)
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 11
})


In [5]:
train_test = dataset.train_test_split(test_size = 0.2, seed = 42)

train_dataset = train_test["train"]
test_dataset = train_test["test"]


In [6]:
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [7]:
def tokenize_function(examples):
    return tokenizer(examples['text'])

tokenized_train_dataset = train_dataset.map(
    tokenize_function,
    batched = True, 
    remove_columns = ['text']
)

tokenized_test_dataset = test_dataset.map(
    tokenize_function,  
    batched = True, 
    remove_columns = ['text']
    )

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [8]:
block_size = 32


def group_texts(examples):

    concatenated_examples = {}

    for key in examples.keys():
        concatenated_examples[key] = sum(
            examples[key],
            []
        )

    total_length = len(
        concatenated_examples["input_ids"]
    )

    total_length = (
        total_length // block_size
    ) * block_size

    result = {}

    for key, tokens in concatenated_examples.items():

        result[key] = [
            tokens[i:i + block_size]
            for i in range(
                0,
                total_length,
                block_size
            )
        ]

    result["labels"] = result["input_ids"].copy()

    return result

In [9]:
lm_train_dataset = tokenized_train_dataset.map(
    group_texts,
    batched=True
)

lm_eval_dataset = tokenized_test_dataset.map(
    group_texts,
    batched=True
)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [10]:
print(
    "Training examples:",
    len(lm_train_dataset)
)

print(
    "Validation examples:",
    len(lm_eval_dataset)
)

Training examples: 7
Validation examples: 2


In [11]:
model = AutoModelForCausalLM.from_pretrained(
    model_name
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [12]:
training_args = TrainingArguments(
    output_dir="./outputs/module9",

    num_train_epochs=3,

    per_device_train_batch_size=2,

    per_device_eval_batch_size=2,

    learning_rate=5e-5,

    logging_steps=1,

    eval_strategy="epoch",

    save_strategy="epoch",

    report_to="none"
)

In [13]:
eval_strategy="epoch"

In [14]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=lm_train_dataset,

    eval_dataset=lm_eval_dataset
)

In [15]:
trainer.train()

c:\Users\Mahesh.shriram\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,4.347780,3.493541
2,1.468620,3.574345
3,0.268307,3.595423


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Mahesh.shriram\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Mahesh.shriram\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12, training_loss=1.9343442420164745, metrics={'train_runtime': 34245.7188, 'train_samples_per_second': 0.001, 'train_steps_per_second': 0.0, 'total_flos': 1443045187584.0, 'train_loss': 1.9343442420164745, 'epoch': 3.0})

In [16]:
trainer.save_model("./outputs/trained_model")
tokenizer.save_pretrained("./outputs/trained_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./outputs/trained_model\\tokenizer_config.json',
 './outputs/trained_model\\chat_template.jinja',
 './outputs/trained_model\\tokenizer.json')

In [17]:
metrics = trainer.evaluate()
print(metrics)

c:\Users\Mahesh.shriram\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch
0.268307,3.595423,3


{'eval_loss': 3.5954227447509766}


In [ ]:
eval_loss = metrics["eval_loss"]

# perplexity:How uncertain the model is when predicting the next token.
# Lower is generally better when comparing models on the same evaluation dataset and tokenization setup.
perplexity = math.exp(eval_loss)
print(
    "Perplexity:",
    perplexity
)

Perplexity: 36.43109778834241


## Before Training Perplexity

In [21]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [22]:
base_trainer = Trainer(
    model=base_model,
    args=training_args,
    eval_dataset=lm_eval_dataset
)

In [23]:
base_matrics = base_trainer.evaluate()
print(base_matrics)

c:\Users\Mahesh.shriram\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch
No log,3.926051,0


{'eval_loss': 3.926050901412964}


In [24]:
base_perplexity = math.exp(base_matrics["eval_loss"])
print(
    "Base Perplexity:", base_perplexity
)

Base Perplexity: 50.70633743041444
